In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime
from subprocess import run
from PIL import Image
import piexif

# ============================================================
# ROOT INPUT
# ============================================================

FRAMES_DIR_ROOT = r"D:\metadata\360\GAGAL\3.frame_360\3"
METADATA_ROOT   = r"D:\metadata\360\GAGAL\1.360_raw\output"
VIDEO_INFO_TXT  = r"D:\metadata\360\GAGAL\3.frame_360\3\summary_txt_fps3.txt"
OUTPUT_ROOT     = r"D:\metadata\360\GAGAL\4.camera_360_geotag"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ============================================================
# LOAD VIDEO DURATIONS
# ============================================================

def load_video_durations(txt_path):
    video_data = {}
    with open(txt_path, "r") as f:
        lines = f.readlines()

    current_name = None
    for line in lines:
        line = line.strip()
        if line.startswith("Nama"):
            current_name = line.split(":")[1].strip().replace(".mp4", "")
        elif line.startswith("Durasi") and current_name:
            dur = float(line.split(":")[1].strip().split()[0])
            video_data[current_name] = dur
            current_name = None

    return video_data

video_durations = load_video_durations(VIDEO_INFO_TXT)
print(f"Durasi ditemukan untuk {len(video_durations)} video")

# ============================================================
# HELPER
# ============================================================

def deg_to_dms_rational(deg_float):
    deg = int(abs(deg_float))
    min_float = (abs(deg_float) - deg) * 60
    minutes = int(min_float)
    seconds = int((min_float - minutes) * 60 * 100)
    return ((deg, 1), (minutes, 1), (seconds, 100))

# ============================================================
# LOOP PER VIDEO
# ============================================================

video_folders = sorted([
    d for d in os.listdir(FRAMES_DIR_ROOT)
    if os.path.isdir(os.path.join(FRAMES_DIR_ROOT, d))
])

for video_id in video_folders:

    print(f"\n=== PROSES VIDEO 360: {video_id} ===")

    FRAMES_DIR = os.path.join(FRAMES_DIR_ROOT, video_id)
    CSV_PATH   = os.path.join(METADATA_ROOT, f"{video_id}.csv")

    if not os.path.exists(CSV_PATH):
        print("Metadata CSV tidak ditemukan, skip")
        continue

    VIDEO_DURATION = video_durations.get(video_id)
    if VIDEO_DURATION is None:
        print("Durasi video tidak ada di TXT, skip")
        continue

    print(f"Durasi video: {VIDEO_DURATION} detik")

    gps_df = pd.read_csv(CSV_PATH)
    gps_df["datetime"] = pd.to_datetime(gps_df["datetime"], errors="coerce")
    gps_df = gps_df.sort_values("t_rel_sec").reset_index(drop=True)

    t_min, t_max = gps_df["t_rel_sec"].min(), gps_df["t_rel_sec"].max()

    img_files = sorted(
        f for f in os.listdir(FRAMES_DIR)
        if re.match(r"^GS\d+_image_\d+\.jpg$", f)
    )

    if not img_files:
        print("Tidak ada frame ditemukan")
        continue

    print(f"Total frame: {len(img_files)}")

    timestamps = np.linspace(0, VIDEO_DURATION, len(img_files))

    sensor_cols = [
        "lat","lon","alt","spd_3d","spd_2d",
        "yaw","pitch","roll",
        "gyro_x","gyro_y","gyro_z",
        "accel_x","accel_y","accel_z","temp"
    ]

    mapping_rows = []

    for img_name, t_vid in zip(img_files, timestamps):

        t_clipped = np.clip(t_vid, t_min, t_max)

        interpolated = {
            col: np.interp(t_clipped, gps_df["t_rel_sec"], gps_df[col])
            for col in sensor_cols if col in gps_df.columns
        }

        nearest_idx = gps_df["t_rel_sec"].sub(t_clipped).abs().idxmin()
        base_datetime = gps_df.loc[nearest_idx, "datetime"]

        row = {
            "image_name": img_name,
            "timestamp_video": t_vid,
            "datetime": base_datetime.isoformat() if pd.notna(base_datetime) else None
        }
        row.update(interpolated)
        mapping_rows.append(row)

    mapping_df = pd.DataFrame(mapping_rows)

    # =====================================================
    # OUTPUT FOLDER PER VIDEO
    # =====================================================
    OUT_VIDEO_DIR = os.path.join(OUTPUT_ROOT, video_id)
    os.makedirs(OUT_VIDEO_DIR, exist_ok=True)

    out_csv = os.path.join(OUT_VIDEO_DIR, f"{video_id}_mapping.csv")
    mapping_df.to_csv(out_csv, index=False)
    print(f"CSV mapping disimpan: {out_csv}")

    # =====================================================
    # WRITE EXIF + XMP
    # =====================================================
    for _, row in mapping_df.iterrows():

        src_path = os.path.join(FRAMES_DIR, row["image_name"])
        dst_path = os.path.join(OUT_VIDEO_DIR, row["image_name"])

        if not os.path.exists(src_path):
            continue

        img = Image.open(src_path)
        exif_dict = {"0th": {}, "Exif": {}, "GPS": {}, "1st": {}, "thumbnail": None}

        lat, lon, alt = row.get("lat"), row.get("lon"), row.get("alt")

        if pd.notna(lat) and pd.notna(lon):
            exif_dict["GPS"][piexif.GPSIFD.GPSLatitudeRef]  = "S" if lat < 0 else "N"
            exif_dict["GPS"][piexif.GPSIFD.GPSLatitude]     = deg_to_dms_rational(lat)
            exif_dict["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
            exif_dict["GPS"][piexif.GPSIFD.GPSLongitude]    = deg_to_dms_rational(lon)

        if pd.notna(alt):
            exif_dict["GPS"][piexif.GPSIFD.GPSAltitude] = (int(alt * 100), 100)

        try:
            dt = datetime.fromisoformat(str(row["datetime"]))
            dt_str = dt.strftime("%Y:%m:%d %H:%M:%S")
            exif_dict["0th"][piexif.ImageIFD.DateTime] = dt_str
            exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal] = dt_str
        except:
            pass

        meta_json = {
            "yaw_deg": row.get("yaw"),
            "pitch_deg": row.get("pitch"),
            "roll_deg": row.get("roll"),
            "spd_3d": row.get("spd_3d"),
            "spd_2d": row.get("spd_2d")
        }

        exif_dict["0th"][piexif.ImageIFD.ImageDescription] = json.dumps(meta_json).encode()

        img.save(dst_path, exif=piexif.dump(exif_dict))

        exiftool_cmd = [
            "exiftool", "-overwrite_original",
            f"-XMP:CameraYaw={row.get('yaw',0)}",
            f"-XMP:PosePitchDegrees={row.get('pitch',0)}",
            f"-XMP:PoseRollDegrees={row.get('roll',0)}",
            dst_path
        ]
        run(exiftool_cmd)

    print(f"Selesai EXIF: {video_id}")

print("\nSEMUA VIDEO 360 SELESAI DIPROSES\n")


Durasi ditemukan untuk 2 video

=== PROSES VIDEO 360: GS014244 ===
Durasi video: 59.86 detik
Total frame: 180
CSV mapping disimpan: D:\metadata\360\GAGAL\4.camera_360_geotag\GS014244\GS014244_mapping.csv
Selesai EXIF: GS014244

=== PROSES VIDEO 360: GS014287 ===
Durasi video: 63.93 detik
Total frame: 192
CSV mapping disimpan: D:\metadata\360\GAGAL\4.camera_360_geotag\GS014287\GS014287_mapping.csv
Selesai EXIF: GS014287

SEMUA VIDEO 360 SELESAI DIPROSES

